# Chapter 1: Data Contracts and Mapping Configs

## Why Your ML Pipeline Breaks When a New Client Sends Data

Every company calls their data something different. One says `MonthlyCharges`, another says `mrr`, a third says `monthly_fee`. They all mean the same thing: how much the customer pays each month.

Without a standard, your pipeline would need custom code for every single client. That's not a product — that's consulting.

This chapter introduces two ideas:

1. **Data Contract** — a universal form that defines what shape data must take. Some fields are mandatory (no entry without them), others are helpful but optional.
2. **Mapping Config** — a per-client Rosetta Stone that translates their column names into our standard language.

Together, these let us onboard ANY client's data with a single YAML file, no code changes.

## The Three-Tier Data Contract

Think of a hospital intake form:

- **Tier 1 (Required):** Name, date of birth, reason for visit — without these, you can't treat the patient.
- **Tier 2 (Helpful):** Allergies, current medications — makes treatment better but you can proceed without them.
- **Tier 3 (Nice-to-have):** Insurance info, emergency contact — useful context, never a dealbreaker.

Our data contract works identically:

| Tier | Fields | Rule |
|------|--------|------|
| 1 (Required) | customer_id, tenure_months, monthly_charges, total_charges, churn_label | Missing ANY = rejected |
| 2 (Engagement) | contract_type, payment_method, support_tickets | Missing = logged, not penalized |
| 3 (Demographics) | gender, age_bucket, partner_status | Missing = noted, never blocks |

In [1]:
# Let's look at the actual schema definition
from churn_pipeline.data_contract import STANDARD_SCHEMA, Tier, FieldSpec, get_tier1_field_names

# The "bouncer's checklist" — these are non-negotiable
print("Tier 1 (Required) fields:")
print("========================")
for name, spec in STANDARD_SCHEMA.items():
    if spec.tier == Tier.REQUIRED:
        print(f"  {name:20s} ({spec.dtype:8s}) — {spec.description[:60]}")

Tier 1 (Required) fields:
  customer_id          (string  ) — Unique customer identifier — who are we talking about?
  tenure_months        (int     ) — How many months this person has been a customer. Longer tenu
  monthly_charges      (float   ) — What the customer pays each month. Higher charges can signal
  total_charges        (float   ) — Cumulative amount billed over the customer's lifetime. Rough
  churn_label          (int     ) — The answer we're trying to predict: 1 = customer left, 0 = c


In [2]:
# Tier 2 and 3 — the optional upgrades
print("Tier 2 (Engagement) fields:")
print("===========================")
for name, spec in STANDARD_SCHEMA.items():
    if spec.tier == Tier.ENGAGEMENT:
        print(f"  {name:20s} ({spec.dtype:8s}) — {spec.description[:60]}")

print("\nTier 3 (Demographics) fields:")
print("=============================")
for name, spec in STANDARD_SCHEMA.items():
    if spec.tier == Tier.DEMOGRAPHIC:
        print(f"  {name:20s} ({spec.dtype:8s}) — {spec.description[:60]}")

Tier 2 (Engagement) fields:
  contract_type        (category) — Contract duration — month-to-month customers leave 3x more o
  payment_method       (category) — How the customer pays. Auto-pay customers tend to be stickie
  support_tickets      (int     ) — Number of support contacts. High ticket counts often signal 

Tier 3 (Demographics) fields:
  gender               (category) — Customer gender. Rarely a strong churn predictor on its own,
  age_bucket           (category) — Age range bucket (e.g., 18-25, 26-35). Different age groups 
  partner_status       (category) — Whether the customer has a partner. Households with shared a


## The Mapping Config: A Rosetta Stone

Every client has their own column names. The mapping config is a YAML file that translates between client-speak and pipeline-speak.

It does three things:
1. **Renames columns** — `MonthlyCharges` becomes `monthly_charges`
2. **Converts values** — `"Yes"` becomes `1`, `"No"` becomes `0`
3. **Fixes types** — a string `"1234.56"` becomes the float `1234.56`

You write this once per client, and then their data flows through the same pipeline as everyone else.

In [3]:
# Load a real mapping config — the IBM Telco dataset
from churn_pipeline.mapping_config import load_mapping_config, apply_mapping

config = load_mapping_config("../configs/client_telco/mapping.yaml")

print(f"Client: {config.client_id}")
print(f"Description: {config.source_description}")
print(f"\nColumn translations ({len(config.column_mappings)} mappings):")
for raw, standard in config.column_mappings.items():
    print(f"  {raw:20s} → {standard}")

Client: telco_ibm
Description: IBM Telco Customer Churn dataset — 7,043 customers with demographics, services, and churn labels

Column translations (9 mappings):
  customerID           → customer_id
  tenure               → tenure_months
  MonthlyCharges       → monthly_charges
  TotalCharges         → total_charges
  Churn                → churn_label
  Contract             → contract_type
  PaymentMethod        → payment_method
  gender               → gender
  Partner              → partner_status


In [4]:
# Value mappings — converting client values to standard ones
print("Value conversions:")
for field, mapping in config.value_mappings.items():
    print(f"\n  {field}:")
    for raw_val, std_val in mapping.items():
        print(f"    {raw_val!r:20s} → {std_val}")

print(f"\nType coercions:")
for field, target in config.type_coercions.items():
    print(f"  {field:20s} → {target}")

Value conversions:

  churn_label:
    'Yes'                → 1
    'No'                 → 0

  contract_type:
    'Month-to-month'     → month-to-month
    'One year'           → one_year
    'Two year'           → two_year

  partner_status:
    'Yes'                → 1
    'No'                 → 0

Type coercions:
  total_charges        → float
  tenure_months        → int


## Applying the Mapping: Before and After

Let's simulate what happens when raw client data hits our mapping config. Watch how the messy, client-specific columns become clean, standardized ones.

In [5]:
import pandas as pd

# Simulate raw IBM Telco data (what the client uploads)
raw_data = pd.DataFrame({
    "customerID": ["7590-VHVEG", "5575-GNVDE", "3668-QPYBK"],
    "tenure": [1, 34, 2],
    "MonthlyCharges": [29.85, 56.95, 53.85],
    "TotalCharges": ["29.85", "1889.5", " "],  # Note: strings! And a blank!
    "Churn": ["No", "Yes", "Yes"],
    "Contract": ["Month-to-month", "One year", "Month-to-month"],
    "Partner": ["Yes", "No", "No"],
})

print("BEFORE mapping (raw client data):")
print("=" * 50)
print(raw_data)
print(f"\nColumn types: {dict(raw_data.dtypes)}")

BEFORE mapping (raw client data):
   customerID  tenure  MonthlyCharges TotalCharges Churn        Contract  \
0  7590-VHVEG       1           29.85        29.85    No  Month-to-month   
1  5575-GNVDE      34           56.95       1889.5   Yes        One year   
2  3668-QPYBK       2           53.85                Yes  Month-to-month   

  Partner  
0     Yes  
1      No  
2      No  

Column types: {'customerID': dtype('O'), 'tenure': dtype('int64'), 'MonthlyCharges': dtype('float64'), 'TotalCharges': dtype('O'), 'Churn': dtype('O'), 'Contract': dtype('O'), 'Partner': dtype('O')}


In [6]:
# Apply the mapping config
mapped_data = apply_mapping(raw_data, config)

print("AFTER mapping (standardized):")
print("=" * 50)
print(mapped_data)
print(f"\nColumn types: {dict(mapped_data.dtypes)}")
print(f"\nNotice:")
print(f"  - 'customerID' became 'customer_id'")
print(f"  - 'Churn' Yes/No became 'churn_label' 1/0")
print(f"  - 'TotalCharges' string became float (the blank ' ' became NaN)")
print(f"  - 'Contract' values normalized to snake_case")

AFTER mapping (standardized):
  customer_id  tenure_months  monthly_charges  total_charges  churn_label  \
0  7590-VHVEG              1            29.85          29.85            0   
1  5575-GNVDE             34            56.95        1889.50            1   
2  3668-QPYBK              2            53.85            NaN            1   

    contract_type  partner_status  
0  month-to-month               1  
1        one_year               0  
2  month-to-month               0  

Column types: {'customer_id': dtype('O'), 'tenure_months': dtype('int64'), 'monthly_charges': dtype('float64'), 'total_charges': dtype('float64'), 'churn_label': dtype('int64'), 'contract_type': dtype('O'), 'partner_status': dtype('int64')}

Notice:
  - 'customerID' became 'customer_id'
  - 'Churn' Yes/No became 'churn_label' 1/0
  - 'TotalCharges' string became float (the blank ' ' became NaN)
  - 'Contract' values normalized to snake_case


## Round-Trip Guarantee

We need to be confident that our mapping configs can be saved and loaded without losing information. This is a "round-trip" property — like translating English to Spanish and back. If the translation is correct, you get back what you started with.

We verify this with property-based testing (Hypothesis), generating hundreds of random configs and checking that serialize → parse → compare always works.

In [7]:
from churn_pipeline.mapping_config import MappingConfig, serialize_mapping_config, load_mapping_config
import tempfile, os

# Demonstrate the round-trip with our telco config
original = load_mapping_config("../configs/client_telco/mapping.yaml")
yaml_str = serialize_mapping_config(original)

# Write to temp file, load back
with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(yaml_str)
    tmp_path = f.name

reloaded = load_mapping_config(tmp_path)
os.unlink(tmp_path)

# Verify
assert reloaded.client_id == original.client_id
assert reloaded.column_mappings == original.column_mappings
print("Round-trip successful!")
print(f"  client_id: {reloaded.client_id}")
print(f"  column_mappings match: {reloaded.column_mappings == original.column_mappings}")
print(f"  value_mappings match: {reloaded.value_mappings == original.value_mappings}")

Round-trip successful!
  client_id: telco_ibm
  column_mappings match: True
  value_mappings match: True


## Three Clients, Three Mapping Configs

To prove this approach works across industries, we have mapping configs for three completely different datasets:

| Client | Industry | Rows | Columns Named Like... |
|--------|----------|------|----------------------|
| telco_ibm | Telecommunications | 7,043 | `MonthlyCharges`, `tenure`, `Churn` |
| ecommerce_d0r1h | E-commerce/SaaS | 37,000 | `CashbackAmount`, `Complain`, `Tenure` |
| banking_churn | Banking | 10,000 | `Balance`, `EstimatedSalary`, `Exited` |

Same pipeline. Different configs. Zero code changes.

In [8]:
# Load all three configs and compare coverage
configs = {
    "telco": load_mapping_config("../configs/client_telco/mapping.yaml"),
    "ecommerce": load_mapping_config("../configs/client_ecommerce/mapping.yaml"),
    "banking": load_mapping_config("../configs/client_banking/mapping.yaml"),
}

print(f"{'Client':<15} {'Columns Mapped':<18} {'Value Maps':<14} {'Type Coercions'}")
print("-" * 60)
for name, cfg in configs.items():
    print(f"{cfg.client_id:<15} {len(cfg.column_mappings):<18} {len(cfg.value_mappings):<14} {len(cfg.type_coercions)}")

Client          Columns Mapped     Value Maps     Type Coercions
------------------------------------------------------------
telco_ibm       9                  3              2
ecommerce_d0r1h 9                  2              4
banking_churn   7                  1              4


## Key Takeaways

1. **Data contracts define the universal shape** — Tier 1 is mandatory, Tier 2/3 are bonus
2. **Mapping configs are per-client translators** — one YAML file handles renaming, value conversion, and type coercion
3. **Round-trip integrity is verified** — configs can be saved/loaded without data loss
4. **Zero code changes per client** — just add a YAML file

Next: Chapter 2 shows how we validate that mapped data actually conforms to the contract before the pipeline proceeds.

---

## Run It For Real: IBM Telco Dataset (7,043 customers)

Everything above used tiny examples to explain the concepts. Now let's do it for real — load the actual IBM Telco dataset, apply our mapping config, and save the result for Chapter 2 to pick up.

**Input:** `data/WA_Fn-UseC_-Telco-Customer-Churn.csv` (raw from IBM)  
**Output:** `data/telco_mapped.csv` (standardized column names, values, and types)

In [ ]:
import os
import urllib.request

# Download the dataset if not already present
DATA_DIR = "../data"
RAW_PATH = os.path.join(DATA_DIR, "WA_Fn-UseC_-Telco-Customer-Churn.csv")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(RAW_PATH):
    url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
    print("Downloading IBM Telco dataset...")
    urllib.request.urlretrieve(url, RAW_PATH)
    print("Done.")
else:
    print(f"Dataset already downloaded: {RAW_PATH}")

# Load it
raw_telco = pd.read_csv(RAW_PATH)
print(f"\nLoaded: {raw_telco.shape[0]} customers, {raw_telco.shape[1]} columns")
print(f"Columns: {raw_telco.columns.tolist()}")
raw_telco.head(3)

In [ ]:
# Apply our mapping config to the FULL dataset
config = load_mapping_config("../configs/client_telco/mapping.yaml")
telco_mapped = apply_mapping(raw_telco, config)

print(f"After mapping:")
print(f"  Shape: {telco_mapped.shape}")
print(f"  Columns: {telco_mapped.columns.tolist()}")
print(f"\nSample (first 5 rows, key columns only):")
key_cols = [c for c in ['customer_id', 'tenure_months', 'monthly_charges', 
            'total_charges', 'churn_label', 'contract_type'] if c in telco_mapped.columns]
telco_mapped[key_cols].head()

In [ ]:
# Save for Chapter 2 to pick up
OUTPUT_PATH = os.path.join(DATA_DIR, "telco_mapped.csv")
telco_mapped.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"  → {telco_mapped.shape[0]} rows, {telco_mapped.shape[1]} columns")
print(f"  → Chapter 2 will load this file and validate it against the data contract.")

---

*Source code: `src/churn_pipeline/data_contract.py`, `src/churn_pipeline/mapping_config.py`*  
*Tests: `tests/property/test_mapping_roundtrip.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*